# 🏆 SHL Grammar Scoring Engine 2026

**Goal:** Predict a continuous MOS Likert grammar score (0–5) from 45–60s spoken WAV files.  
**Strategy:** Three-Tower Ensemble → Whisper ASR + DeBERTa-v3-base · HuBERT-large MLP · LightGBM  
**Metric:** RMSE ↓ · Pearson ↑  
**Target:** RMSE < 0.32 (beat leaderboard rank 1 at 0.3239)

```
Dataset_Final/
├── train/   (769 .wav files)
├── test/    (216 .wav files)
├── train.csv           → filename, label
├── test.csv            → filename
└── sample_submission.csv
```

## ① Install Dependencies

In [ ]:
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("openai-whisper", "language-tool-python", "lightgbm", "librosa", "soundfile",
    "transformers[torch]", "accelerate", "sentencepiece", "scipy", "tqdm")

## ② Imports & Paths

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import whisper
import librosa
import language_tool_python
import lightgbm as lgb
from pathlib import Path
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Wav2Vec2FeatureExtractor, HubertModel,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings("ignore")
SEED   = 42
N_FOLD = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device : {DEVICE}")
if DEVICE == "cuda": print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Exact dataset paths ────────────────────────────────────────────────────────
ROOT       = Path("/kaggle/input/shl-hiring-assessment-2026/Dataset_Final")
TRAIN_AUDIO= ROOT / "train"          # 769 wav files
TEST_AUDIO = ROOT / "test"           # 216 wav files
WORK_DIR   = Path("/kaggle/working")

assert ROOT.exists(),        f"Root not found: {ROOT}"
assert TRAIN_AUDIO.exists(), f"Train audio dir not found: {TRAIN_AUDIO}"
assert TEST_AUDIO.exists(),  f"Test audio dir not found: {TEST_AUDIO}"
print(f"Train WAVs : {len(list(TRAIN_AUDIO.glob('*.wav')))}")
print(f"Test  WAVs : {len(list(TEST_AUDIO.glob('*.wav')))}")

## ③ Load CSVs & Inspect

In [ ]:
train_df = pd.read_csv(ROOT / "train.csv")
test_df  = pd.read_csv(ROOT / "test.csv")
sample   = pd.read_csv(ROOT / "sample_submission.csv")

print("train.csv columns :", train_df.columns.tolist())
print("test.csv  columns :", test_df.columns.tolist())
print("sample_submission :", sample.columns.tolist())
print()
print(train_df.head(3))
print()
print(f"Label range : [{train_df.iloc[:,1].min():.2f}, {train_df.iloc[:,1].max():.2f}]")
print(f"Label mean  : {train_df.iloc[:,1].mean():.3f} ± {train_df.iloc[:,1].std():.3f}")

# Auto-detect column names from sample_submission
FILE_COL  = sample.columns[0]   # e.g. 'filename'
SCORE_COL = sample.columns[1]   # e.g. 'label'
LABEL_COL = train_df.columns[1] # label column in train.csv
print(f"\nFILE_COL={FILE_COL!r}  SCORE_COL={SCORE_COL!r}  LABEL_COL={LABEL_COL!r}")

def resolve_wav(fname, split="train"):
    """Return full path to wav; handles filenames with/without .wav extension."""
    audio_dir = TRAIN_AUDIO if split == "train" else TEST_AUDIO
    f = str(fname)
    p = audio_dir / (f if f.endswith(".wav") else f + ".wav")
    return p

## ④ Tower A — ASR: Whisper large-v3
State-of-the-art open-source ASR. Results cached to disk after first run.

In [ ]:
TRANS_CACHE = WORK_DIR / "transcriptions.csv"

if TRANS_CACHE.exists():
    trans_df = pd.read_csv(TRANS_CACHE)
    print(f"✓ Loaded cached transcriptions ({len(trans_df)} rows)")
else:
    print("Loading Whisper large-v3 …")
    wmodel = whisper.load_model("large-v3", device=DEVICE)

    rows = []
    for split, df in [("train", train_df), ("test", test_df)]:
        for fname in tqdm(df[FILE_COL], desc=f"Whisper [{split}]"):
            path = resolve_wav(fname, split)
            try:
                r = wmodel.transcribe(
                    str(path), fp16=(DEVICE=="cuda"), language="en",
                    condition_on_previous_text=False, temperature=0
                )
                text = r["text"].strip()
            except Exception as e:
                print(f"  ✗ {fname}: {e}")
                text = ""
            rows.append({FILE_COL: fname, "split": split, "transcript": text})

    trans_df = pd.DataFrame(rows)
    trans_df.to_csv(TRANS_CACHE, index=False)
    del wmodel; torch.cuda.empty_cache()
    print(f"✓ Transcription done → {TRANS_CACHE}")

train_trans = trans_df[trans_df["split"]=="train"][[FILE_COL, "transcript"]]
test_trans  = trans_df[trans_df["split"]=="test"][[FILE_COL, "transcript"]]

train_df = train_df.merge(train_trans, on=FILE_COL, how="left")
test_df  = test_df.merge(test_trans,   on=FILE_COL, how="left")
train_df["transcript"] = train_df["transcript"].fillna("").astype(str)
test_df["transcript"]  = test_df["transcript"].fillna("").astype(str)

print("Sample train transcript:", train_df["transcript"].iloc[0][:120])

## ⑤ Tower B — HuBERT-large Audio Embeddings
Extracts 1024-dim acoustic embeddings encoding fluency, prosody, and disfluency — signals lost by ASR.

In [ ]:
HUBERT_CACHE = WORK_DIR / "hubert_embeddings.npy"
HUBERT_META  = WORK_DIR / "hubert_meta.json"

if HUBERT_CACHE.exists():
    emb_all   = np.load(HUBERT_CACHE)
    meta      = json.load(open(HUBERT_META))
    n_tr      = meta["n_train"]
    print(f"✓ Loaded cached HuBERT embeddings {emb_all.shape}")
else:
    print("Loading HuBERT-large …")
    feat_ext = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-large-ls960-ft")
    hubert   = HubertModel.from_pretrained("facebook/hubert-large-ls960-ft").to(DEVICE)
    hubert.eval()

    def embed_wav(path):
        try:
            y, _ = librosa.load(str(path), sr=16000, mono=True)
            y    = y[:16000*62]                     # cap at 62 s
            inp  = feat_ext(y, sampling_rate=16000, return_tensors="pt", padding=True)
            inp  = {k: v.to(DEVICE) for k, v in inp.items()}
            with torch.no_grad():
                out = hubert(**inp).last_hidden_state  # (1, T, 1024)
            return out.mean(dim=1).squeeze().cpu().numpy()
        except Exception as e:
            print(f"  ✗ {path.name}: {e}")
            return np.zeros(1024)

    tr_embs = [embed_wav(resolve_wav(f, "train")) for f in tqdm(train_df[FILE_COL], desc="HuBERT [train]")]
    te_embs = [embed_wav(resolve_wav(f, "test"))  for f in tqdm(test_df[FILE_COL],  desc="HuBERT [test]")]

    n_tr    = len(tr_embs)
    emb_all = np.vstack(tr_embs + te_embs)
    np.save(HUBERT_CACHE, emb_all)
    json.dump({"n_train": n_tr}, open(HUBERT_META, "w"))
    del hubert; torch.cuda.empty_cache()
    print(f"✓ HuBERT done → {emb_all.shape}")

hubert_train = emb_all[:n_tr]
hubert_test  = emb_all[n_tr:]
print(f"Train: {hubert_train.shape}  Test: {hubert_test.shape}")

## ⑥ Tower C — Handcrafted Features
LanguageTool grammar errors + librosa acoustic statistics (MFCC, pitch, tempo, energy).

In [ ]:
FEAT_CACHE = WORK_DIR / "handcrafted_features.npy"

if FEAT_CACHE.exists():
    feat_all = np.load(FEAT_CACHE)
    print(f"✓ Loaded cached features {feat_all.shape}")
else:
    lt = language_tool_python.LanguageTool("en-US")

    def text_feats(text):
        text  = str(text).strip()
        if not text: return np.zeros(10)
        words = text.split()
        n     = max(len(words), 1)
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        matches = lt.check(text)
        n_err   = len(matches)
        gram_e  = sum(1 for m in matches if "GRAMMAR" in m.ruleId.upper())
        spell_e = sum(1 for m in matches if "SPELL"   in m.ruleId.upper())
        punct   = sum(1 for c in text if c in ".,;:!?")
        caps    = sum(1 for w in words if w and w[0].isupper())
        unique  = len(set(w.lower() for w in words))
        return np.array([
            n_err / n, gram_e / n, spell_e / n,
            n, len(sents),
            np.mean([len(w) for w in words]),
            unique / n,           # type-token ratio
            unique,
            punct / n,
            caps  / n,
        ], dtype=float)

    def audio_feats(path):
        try:
            y, sr = librosa.load(str(path), sr=16000, mono=True)
            mfcc  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
            pitch, _ = librosa.piptrack(y=y, sr=sr)
            pv    = pitch[pitch > 0]
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            rms   = librosa.feature.rms(y=y)[0]
            zcr   = librosa.feature.zero_crossing_rate(y)[0]
            spec  = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            return np.concatenate([
                mfcc.mean(1), mfcc.std(1),                  # 40
                [pv.mean() if len(pv) else 0,
                 pv.std()  if len(pv) else 0,
                 float(tempo), rms.mean(), rms.std(),
                 zcr.mean(), zcr.std(),
                 spec.mean(), spec.std(),
                 len(y)/sr],                                 # 10
            ])
        except:
            return np.zeros(50)

    print("Extracting text features …")
    tr_txt = np.stack([text_feats(t) for t in tqdm(train_df["transcript"])])
    te_txt = np.stack([text_feats(t) for t in tqdm(test_df["transcript"])])

    print("Extracting audio features …")
    tr_aud = np.stack([audio_feats(resolve_wav(f,"train")) for f in tqdm(train_df[FILE_COL], desc="librosa train")])
    te_aud = np.stack([audio_feats(resolve_wav(f,"test"))  for f in tqdm(test_df[FILE_COL],  desc="librosa test")])

    lt.close()
    feat_all = np.vstack([
        np.hstack([tr_txt, tr_aud]),
        np.hstack([te_txt, te_aud]),
    ])
    np.save(FEAT_CACHE, feat_all)
    print(f"✓ Features done → {feat_all.shape}")

hand_train = feat_all[:len(train_df)]
hand_test  = feat_all[len(train_df):]

## ⑦ Tower A: DeBERTa-v3-base Fine-Tuning (5-Fold CV)
Fine-tune a regression head on Whisper transcripts. OOF predictions act as meta-features.

In [ ]:
MODEL_NAME  = "microsoft/deberta-v3-base"
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
labels_all  = train_df[LABEL_COL].values.astype(float)
texts_train = train_df["transcript"].tolist()
texts_test  = test_df["transcript"].tolist()

class TxtDS(Dataset):
    def __init__(self, texts, labels=None, max_len=256):
        self.enc = tokenizer(texts, padding="max_length", truncation=True,
                             max_length=max_len, return_tensors="pt")
        self.y   = labels
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        if self.y is not None:
            d["labels"] = torch.tensor(float(self.y[i]), dtype=torch.float)
        return d

def metrics_fn(ep):
    p, y = ep
    p = np.clip(p.squeeze(), 0, 5)
    return {"rmse": float(np.sqrt(mean_squared_error(y, p))),
            "pearson": float(pearsonr(y, p)[0])}

def fresh_model():
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, ignore_mismatched_sizes=True)
    m.classifier.dropout.p = 0.2
    return m

TARGS = TrainingArguments(
    output_dir                  = str(WORK_DIR/"deb_ckpt"),
    num_train_epochs            = 8,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1.5e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.15,
    lr_scheduler_type           = "cosine",
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "pearson",
    greater_is_better           = True,
    fp16                        = (DEVICE=="cuda"),
    gradient_accumulation_steps = 2,
    report_to                   = "none",
    seed                        = SEED,
    save_total_limit            = 1,
    dataloader_num_workers      = 2,
)

kf            = KFold(n_splits=N_FOLD, shuffle=True, random_state=SEED)
oof_deberta   = np.zeros(len(train_df))
test_deberta  = np.zeros(len(test_df))

for fold, (tri, vli) in enumerate(kf.split(texts_train)):
    print(f"\n{'─'*55}  Fold {fold+1}/{N_FOLD}")
    tr_ds  = TxtDS([texts_train[i] for i in tri],  labels_all[tri])
    vl_ds  = TxtDS([texts_train[i] for i in vli],  labels_all[vli])
    te_ds  = TxtDS(texts_test)

    t = Trainer(model=fresh_model(), args=TARGS,
                train_dataset=tr_ds, eval_dataset=vl_ds,
                compute_metrics=metrics_fn,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    t.train()

    oof_deberta[vli]  = t.predict(vl_ds).predictions.squeeze()
    test_deberta     += t.predict(te_ds).predictions.squeeze() / N_FOLD

    vl  = np.clip(oof_deberta[vli], 0, 5)
    r,_ = pearsonr(labels_all[vli], vl)
    print(f"  → RMSE={np.sqrt(mean_squared_error(labels_all[vli],vl)):.4f}  Pearson={r:.4f}")
    del t; torch.cuda.empty_cache()

c   = np.clip(oof_deberta, 0, 5)
r,_ = pearsonr(labels_all, c)
print(f"\n{'='*55}")
print(f"DeBERTa OOF  RMSE    : {np.sqrt(mean_squared_error(labels_all,c)):.4f}")
print(f"DeBERTa OOF  Pearson : {r:.4f}")

## ⑧ Tower B: HuBERT MLP Regressor (5-Fold CV)
Shallow MLP on 1024-dim HuBERT embeddings. Captures acoustic fluency signals.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d=1024, h=512, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, h),   nn.GELU(), nn.Dropout(drop),
            nn.Linear(h, 256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp(Xtr, ytr, Xvl, yvl, epochs=100):
    m   = MLP().to(DEVICE)
    opt = AdamW(m.parameters(), lr=3e-4, weight_decay=1e-2)
    sch = CosineAnnealingLR(opt, T_max=epochs)
    lf  = nn.MSELoss()
    dl  = DataLoader(
        list(zip(torch.tensor(Xtr, dtype=torch.float32),
                 torch.tensor(ytr, dtype=torch.float32))),
        batch_size=32, shuffle=True)
    best_p, best_r = None, 9999
    for ep in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); lf(m(xb), yb).backward(); opt.step()
        sch.step()
        m.eval()
        with torch.no_grad():
            vp = m(torch.tensor(Xvl, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        r = np.sqrt(mean_squared_error(yvl, np.clip(vp,0,5)))
        if r < best_r: best_r, best_p = r, vp.copy()
    return m, best_p

oof_hub  = np.zeros(len(train_df))
test_hub = np.zeros(len(test_df))

for fold, (tri, vli) in enumerate(kf.split(hubert_train)):
    print(f"HuBERT MLP  Fold {fold+1}/{N_FOLD} …", end=" ")
    m, vp = train_mlp(hubert_train[tri], labels_all[tri],
                      hubert_train[vli], labels_all[vli])
    oof_hub[vli] = vp
    m.eval()
    with torch.no_grad():
        tp = m(torch.tensor(hubert_test, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    test_hub += tp / N_FOLD
    r = np.sqrt(mean_squared_error(labels_all[vli], np.clip(vp,0,5)))
    print(f"RMSE={r:.4f}")
    del m; torch.cuda.empty_cache()

c2  = np.clip(oof_hub, 0, 5)
r2,_= pearsonr(labels_all, c2)
print(f"\nHuBERT MLP OOF  RMSE    : {np.sqrt(mean_squared_error(labels_all,c2)):.4f}")
print(f"HuBERT MLP OOF  Pearson : {r2:.4f}")

## ⑨ Tower C: LightGBM on Handcrafted + HuBERT-PCA (5-Fold CV)

In [ ]:
# Reduce 1024-dim HuBERT to 64 PCA components for LGB
pca = PCA(n_components=64, random_state=SEED)
Xtr_lgb = np.hstack([hand_train, pca.fit_transform(hubert_train)])
Xte_lgb = np.hstack([hand_test,  pca.transform(hubert_test)])

sc = StandardScaler()
Xtr_lgb = sc.fit_transform(Xtr_lgb)
Xte_lgb = sc.transform(Xte_lgb)

lgb_params = dict(
    objective="regression_l1", metric="rmse",
    n_estimators=3000, learning_rate=0.02,
    num_leaves=31, min_child_samples=8,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.05, reg_lambda=0.1,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb  = np.zeros(len(Xtr_lgb))
test_lgb = np.zeros(len(Xte_lgb))

for fold, (tri, vli) in enumerate(kf.split(Xtr_lgb)):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(Xtr_lgb[tri], labels_all[tri],
          eval_set=[(Xtr_lgb[vli], labels_all[vli])],
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(500)])
    oof_lgb[vli]  = m.predict(Xtr_lgb[vli])
    test_lgb     += m.predict(Xte_lgb) / N_FOLD
    r = np.sqrt(mean_squared_error(labels_all[vli], np.clip(oof_lgb[vli],0,5)))
    print(f"LGB Fold {fold+1}: RMSE={r:.4f}")

c3  = np.clip(oof_lgb, 0, 5)
r3,_= pearsonr(labels_all, c3)
print(f"\nLightGBM OOF  RMSE    : {np.sqrt(mean_squared_error(labels_all,c3)):.4f}")
print(f"LightGBM OOF  Pearson : {r3:.4f}")

## ⑩ Optimal Ensemble via Nelder-Mead (OOF-based, no leakage)
Minimize OOF RMSE to find the best weights across all three towers.

In [ ]:
OOF  = np.column_stack([oof_deberta, oof_hub, oof_lgb])
TEST = np.column_stack([test_deberta, test_hub, test_lgb])

def blend_rmse(w):
    w = np.abs(w) / (np.abs(w).sum() + 1e-9)
    return float(np.sqrt(mean_squared_error(labels_all, np.clip(OOF @ w, 0, 5))))

res   = minimize(blend_rmse, x0=[0.5, 0.3, 0.2], method="Nelder-Mead",
                 options={"maxiter": 10000, "xatol": 1e-7})
raw_w = np.abs(res.x)
opt_w = raw_w / raw_w.sum()

print(f"Optimal weights → DeBERTa: {opt_w[0]:.3f}  HuBERT: {opt_w[1]:.3f}  LGB: {opt_w[2]:.3f}")

oof_final  = np.clip(OOF  @ opt_w, 0, 5)
test_final = np.clip(TEST @ opt_w, 0, 5)

final_rmse = np.sqrt(mean_squared_error(labels_all, oof_final))
final_r, _ = pearsonr(labels_all, oof_final)

print()
print("╔═══════════════════════════════════════════════╗")
print(f"║  DeBERTa-v3-base  OOF RMSE : {np.sqrt(mean_squared_error(labels_all,np.clip(oof_deberta,0,5))):.4f}          ║")
print(f"║  HuBERT-large     OOF RMSE : {np.sqrt(mean_squared_error(labels_all,np.clip(oof_hub,0,5))):.4f}          ║")
print(f"║  LightGBM         OOF RMSE : {np.sqrt(mean_squared_error(labels_all,np.clip(oof_lgb,0,5))):.4f}          ║")
print(f"║  ─────────────────────────────────────────  ║")
print(f"║  FINAL Ensemble   OOF RMSE : {final_rmse:.4f}          ║")
print(f"║  FINAL Ensemble   Pearson  : {final_r:.4f}          ║")
print("╚═══════════════════════════════════════════════╝")

## ⑪ Generate submission.csv

In [ ]:
# Match exact format of sample_submission.csv
submission = pd.DataFrame({
    FILE_COL  : test_df[FILE_COL].values,
    SCORE_COL : test_final,
})

OUT = WORK_DIR / "submission.csv"
submission.to_csv(OUT, index=False)

print(f"✅  submission.csv → {OUT}")
print(f"Rows: {len(submission)}  |  "
      f"Score range: [{test_final.min():.3f}, {test_final.max():.3f}]  |  "
      f"Mean: {test_final.mean():.3f}")
print()
print(submission.head(10).to_string(index=False))